<a href="https://colab.research.google.com/github/watch-duty/radio-transcription/blob/main/model/colabs/create_nemo_manifest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Use gemini for transcribing a provided broadcastify mp3 link

In [ ]:

from google import genai
from google.genai import types

API_KEY = 'foo' #@param
BROADCASTIFY_URL = "https://archives.broadcastify.com/141/20260114/202601140230-26474-141.mp3" #@param see archive_urls_sample_20260114_12hrs.csv


In [19]:
client = genai.Client(api_key=API_KEY)

def main():
  prompt = """
    Process the audio file and generate a detailed transcription.
    This audio recording captures a series of radio communications between units and dispatcher 
    They will often use emergency response shorthands and abbreviations.
    The language is in English.

    Requirements:
    1. Identify distinct speakers (e.g., Speaker 1, Speaker 2, or names if context allows).
    2. Provide accurate timestamps for each segment (Format: MM:SS).
    3. Provide a brief summary of the entire audio at the beginning.
    4. Please print the output in a human readable format, with new lines.
  """

  response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=[
      types.Content(
        parts=[
          types.Part(
            file_data=types.FileData(
              file_uri=BROADCASTIFY_URL
            )
          ),
          types.Part(
            text=prompt
          )
        ]
      )
    ],
    config=types.GenerateContentConfig(
      response_mime_type="application/json",
      response_schema=types.Schema(
        type=types.Type.OBJECT,
        properties={
          "summary": types.Schema(
            type=types.Type.STRING,
            description="A concise summary of the audio content.",
          ),
          "segments": types.Schema(
            type=types.Type.ARRAY,
            description="List of transcribed segments with speaker and timestamp.",
            items=types.Schema(
              type=types.Type.OBJECT,
              properties={
                "speaker": types.Schema(type=types.Type.STRING),
                "timestamp": types.Schema(type=types.Type.STRING),
                "content": types.Schema(type=types.Type.STRING),
              },
              required=["speaker", "timestamp", "content"],
            ),
          ),
        },
        required=["summary", "segments"],
      ),
    ),
  )
  print(response.text)

if __name__ == "__main__":
  main()

{
  "summary": "This recording captures various public safety radio transmissions. It begins with a unit from the Arapahoe County Sheriff's Office identifying itself and its location, followed by several ambulance units from Northglenn and South Metro reporting their status as clear and returning to service.",
  "segments": [
    {
      "speaker": "Speaker 1",
      "timestamp": "00:00",
      "content": "WGW-240, Arapahoe County Sheriff, unit 303, Littleton 12. Resting on Littleton 12."
    },
    {
      "speaker": "Speaker 2",
      "timestamp": "00:09",
      "content": "Northglenn Ambulance 2."
    },
    {
      "speaker": "Speaker 3",
      "timestamp": "00:12",
      "content": "Clear and back in service."
    },
    {
      "speaker": "Speaker 4",
      "timestamp": "00:16",
      "content": "South Metro Ambulance 2, clear returning in service."
    }
  ]
}


## Disclaimer: I don't know if this is accurate lol